In [7]:
import pandas as pd
import numpy as np
import re

from datetime import datetime
from pathlib import Path

from sqlalchemy.orm import Session
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

In [8]:
import sys

ROOT_DIR = r"D:\ESM"

if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print(ROOT_DIR)

D:\ESM


In [3]:
from app.core.config import settings
from app.core.database import Base

from app.models.user import User
from app.models.department import Department
from app.models.role import UserRole
from app.models.user_type import UserType
from app.models.mst_status import Status

from app.models.risk_register import RiskRegister
from app.models.risk_description import RiskDescription
from app.models.risk_treatment import RiskTreatment

from app.services.risk_service import get_color_code

In [4]:
# Excel Path

EXCEL_FILE = r"D:\ESM\Data\1. HR- Risk Register Review_2025_Final.xlsx"

# Department

DEFAULT_DEPT_ID = 20

# Demo Password

DEFAULT_PASSWORD = "123456"

# Created By

CREATED_BY = 1

# Financial Year

FINANCIAL_YEAR = "2026-2027"

# Risk Status

RISK_STATUS = 9

# Approval Status

APPROVAL_STATUS = 12

In [5]:
engine = create_engine(settings.DATABASE_URL)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

db = SessionLocal()

print("Database Connected")

Database Connected


In [6]:
raw_df = pd.read_excel(EXCEL_FILE)

raw_df.head(10)

,S. No,Category,Risk Description,Inherent Risk Level,Current Mitigation,Current\nRisk Level,Risk Owner,Risk Treatment,Unnamed: 8,Unnamed: 9,Q1,Q2,Q3,Q4,Back up document
0,NaN,NaN,NaN,(Impact/ Likelihood),NaN,(Impact/ Likelihood),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Action Plan,Due Date,Action owner,NaN,NaN,NaN,NaN,NaN
2,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,5.0,Compensation & Benefits,Risk of overcompensation or undercompensation ...,3D,1. Periodic benchmark of compensation against ...,2C,Shirin V.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
raw_df.columns = [
    "S.No",
    "Category",
    "Risk Description",
    "Inherent Risk",
    "Current Mitigation",
    "Current Risk",
    "Risk Owner",
    # "Risk Treatment",
    "Action Plan",
    "Due Date",
    "Action Owner",
    "Q1",
    "Q2",
    "Q3",
    "Q4",
    "Backup"
]

In [8]:
df = raw_df.iloc[2:].copy()

df.reset_index(drop=True, inplace=True)

df.head()

,S.No,Category,Risk Description,Inherent Risk,Current Mitigation,Current Risk,Risk Owner,Action Plan,Due Date,Action Owner,Q1,Q2,Q3,Q4,Backup
0,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df["Category"] = df["Category"].ffill()

df["S.No"] = df["S.No"].ffill()

In [10]:
text_columns = [
    "Category",
    "Risk Description",
    "Current Mitigation",
    "Risk Owner",
    "Action Plan",
    "Action Owner"
]

for col in text_columns:

    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

In [11]:
df.head(10)

,S.No,Category,Risk Description,Inherent Risk,Current Mitigation,Current Risk,Risk Owner,Action Plan,Due Date,Action Owner,Q1,Q2,Q3,Q4,Backup
0,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,,NaN,,NaN,NaN,NaN,NaN,NaN
1,1.0,Recruitment & Hiring,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
2,1.0,Recruitment & Hiring,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
3,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,,NaN,,NaN,NaN,NaN,NaN,NaN
4,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
5,5.0,Compensation & Benefits,Risk of overcompensation or undercompensation ...,3D,1. Periodic benchmark of compensation against ...,2C,Shirin V.,,NaN,,NaN,NaN,NaN,NaN,NaN


In [12]:
df.groupby(["S.No", "Category"]).size()

S.No  Category               
1.0   Recruitment & Hiring       3
2.0   Employee Exp.              1
4.0   Complaiance Mgt.           1
5.0   Compensation & Benefits    1
dtype: int64

In [13]:
grouped = df.groupby(["S.No", "Category"])

for (sno, category), group in grouped:

    print("=" * 50)
    print("Risk :", category)

    print(group[[
        "Risk Description",
        "Risk Owner"
    ]])

Risk : Recruitment & Hiring
                                    Risk Description Risk Owner
0  Delay  in acquisition of key Positions leading...  Shruti N.
1                       Inadequate background checks     Manali
2  Attrition risk : Inability to retain people at...     Manali
Risk : Employee Exp.
                                    Risk Description  Risk Owner
3  Employee Disengagement and Inadequate talent D...  Abhijit S.
Risk : Complaiance Mgt.
                                    Risk Description Risk Owner
4  Non-compliance to applicble labour laws leadin...     Manali
Risk : Compensation & Benefits
                                    Risk Description Risk Owner
5  Risk of overcompensation or undercompensation ...  Shirin V.


In [14]:
risk = {
    "category": category,
    "rows": group
}

In [15]:
def clean_text(value):
    """
    Convert NaN to empty string and trim spaces.
    """

    if pd.isna(value):
        return ""

    return str(value).strip()


def split_name(full_name):
    """
    Split full name into first name and last name.
    """

    full_name = clean_text(full_name)

    if not full_name:
        return "", ""

    parts = full_name.split()

    first_name = parts[0]

    last_name = " ".join(parts[1:]) if len(parts) > 1 else ""

    return first_name, last_name


def create_log_id(email):
    """
    Use email as login id.
    """

    email = clean_text(email)

    return email.lower()


def clean_email(email):
    """
    Remove spaces and convert to lowercase.
    """

    return clean_text(email).lower()

In [16]:
impact_reverse = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5
}


def parse_risk_code(code):

    code = clean_text(code).upper()

    if code == "":
        return None, None

    match = re.match(r"([1-5])([A-E])", code)

    if not match:
        raise ValueError(f"Invalid Risk Code : {code}")

    likelihood = int(match.group(1))

    impact = impact_reverse[match.group(2)]

    return likelihood, impact

In [17]:
impact_map = {
    1: "A",
    2: "B",
    3: "C",
    4: "D",
    5: "E"
}


def build_color_code(likelihood, impact):

    if likelihood is None or impact is None:
        return None

    return f"{likelihood}{impact_map.get(impact)}"


def get_color(code):
    """
    Wrapper for your existing function.
    """

    return get_color_code(code)

In [18]:
def parse_date(value):

    if pd.isna(value):
        return None

    if isinstance(value, datetime):
        return value

    return pd.to_datetime(value)


def current_time():

    return datetime.now()

In [19]:
def generate_risk_id(db: Session, dept_id: int):

    dept = (
        db.query(Department)
        .filter(Department.id == dept_id)
        .with_for_update()
        .first()
    )

    if not dept:
        raise Exception("Department not found")

    dept.last_risk_number += 1

    number = dept.last_risk_number

    risk_id = f"{dept.dept_short_name}-{str(number).zfill(4)}"

    return risk_id

In [20]:
print(parse_date("2026-07-10"))

print(current_time())

print(generate_risk_id(db, DEFAULT_DEPT_ID))

2026-07-10 00:00:00
2026-07-02 10:25:24.734857
HR-2391


Make User Data

In [21]:
def get_excel_users(df):
    """
    Prepare unique users from Excel.
    """

    users = []

    # Risk Owners
    for name in df["Risk Owner"].dropna().unique():

        name = clean_text(name)

        if name:
            users.append({
                "name": name,
                "email": "",
                "user_type": "Risk Owner",
                "dept_id": DEFAULT_DEPT_ID
            })

    # Action Owners
    if "Action Owner" in df.columns:

        for name in df["Action Owner"].dropna().unique():

            name = clean_text(name)

            if name:

                users.append({
                    "name": name,
                    "email": "",
                    "user_type": "Action Owner",
                    "dept_id": DEFAULT_DEPT_ID
                })

    # Remove duplicate users
    unique_users = {}

    for user in users:
        unique_users[user["name"].lower()] = user

    return list(unique_users.values())

In [22]:
excel_users = get_excel_users(df)

len(excel_users)

4

In [23]:
excel_users

[{'name': 'Shruti N.', 'email': '', 'user_type': 'Risk Owner', 'dept_id': 20},
 {'name': 'Manali', 'email': '', 'user_type': 'Risk Owner', 'dept_id': 20},
 {'name': 'Abhijit S.', 'email': '', 'user_type': 'Risk Owner', 'dept_id': 20},
 {'name': 'Shirin V.', 'email': '', 'user_type': 'Risk Owner', 'dept_id': 20}]

In [24]:
def find_user(db, email=None, full_name=None):

    # Search by Email
    if email:

        user = (
            db.query(User)
            .filter(
                User.email.ilike(email),
                User.is_deleted == 0
            )
            .first()
        )

        if user:
            return user

    # Search by Name
    if full_name:

        first_name, last_name = split_name(full_name)

        user = (
            db.query(User)
            .filter(
                User.first_name.ilike(first_name),
                User.last_name.ilike(last_name),
                User.is_deleted == 0
            )
            .first()
        )

        if user:
            return user

    return None

In [25]:
roles = {
    role.name.strip().lower(): role.id
    for role in db.query(UserRole).filter(UserRole.is_deleted == 0).all()
}

In [26]:
user_types = {
    user_type.name.strip().lower(): user_type.id
    for user_type in db.query(UserType).filter(UserType.is_deleted == 0).all()
}

In [27]:
def prepare_user_data(user_data):

    if user_data["user_type"] == "Risk Owner":

        role_id = roles["risk_owner"]
        user_type_id = user_types["risk owner"]

    elif user_data["user_type"] == "Action Owner":

        role_id = roles["action_owner"]
        user_type_id = user_types["action owner"]

    elif user_data["user_type"] == "Functional Head":

        role_id = roles["functional_head"]
        user_type_id = user_types["functional head"]

    else:
        raise Exception(f"Unknown User Type : {user_data['user_type']}")

    first_name, last_name = split_name(user_data["name"])

    return {

        "log_id": user_data["email"] or first_name.lower(),

        "password": "123456",

        "first_name": first_name,

        "last_name": last_name,

        "email": user_data["email"] if user_data["email"] else f"{first_name.lower()}@example.com",

        "dept_id": user_data["dept_id"],

        "role_id": role_id,

        "user_type_id": user_type_id,

        "status": "Active",

        "created_by": 1,

        "created_on": datetime.utcnow(),

        "is_deleted": 0

    }

In [28]:
def create_user(db, user_dict):

    user = User(**user_dict)

    db.add(user)

    db.flush()

    db.refresh(user)

    print(f"Created : {user.first_name}")

    return user

In [29]:
from unicodedata import name


def import_users(db, excel_users):

    user_map = {}

    print("=" * 60)
    print("IMPORTING USERS")
    print("=" * 60)

    for user_data in excel_users:

        print(f"\nChecking : {user_data['name']}")

        # Step 1 - Find User
        user = find_user(
            db=db,
            email=user_data["email"],
            full_name=user_data["name"]
        )

        # Step 2 - Create User if Missing
        if user is None:

            print("Not Found")

            user_dict = prepare_user_data(user_data)

            user = create_user(
                db=db,
                user_dict=user_dict
            )

        else:

            print(f"Already Exists (ID : {user.id})")

        # Step 3 - Store User
        user_map[user_data["name"]] = {
                "id": user.id,
                "name": f"{user.first_name} {user.last_name}".strip(),
                "email": user.email
            }

    print("\nUser Import Completed")

    return user_map

In [30]:
excel_users = get_excel_users(df)

user_map = import_users(
    db=db,
    excel_users=excel_users
)

IMPORTING USERS

Checking : Shruti N.
Already Exists (ID : 122)

Checking : Manali
Already Exists (ID : 123)

Checking : Abhijit S.
Already Exists (ID : 124)

Checking : Shirin V.
Already Exists (ID : 125)

User Import Completed


In [31]:
try:
    excel_users = get_excel_users(df)

    user_map = import_users(db, excel_users)

    db.commit()

    print("✅ Import completed successfully.")

except Exception as e:
    db.rollback()
    print(f"❌ Import failed: {e}")
    raise

finally:
    db.close()

IMPORTING USERS

Checking : Shruti N.
Already Exists (ID : 122)

Checking : Manali
Already Exists (ID : 123)

Checking : Abhijit S.
Already Exists (ID : 124)

Checking : Shirin V.
Already Exists (ID : 125)

User Import Completed
✅ Import completed successfully.


Import Risk Register

In [32]:
DEFAULT_RISK_STATUS = 9
DEFAULT_FIN_YEAR = '2026-2027'

In [33]:
from app.schemas.risk_register import RiskRegisterCreate

def prepare_risk_register_data(
    db,
    row,
    user_map
):

    risk_owner_id = user_map[row["Risk Owner"]]["id"]

    risk = RiskRegisterCreate(

        risk_name=clean_text(row["Category"]),

        dept_id=DEFAULT_DEPT_ID,

        risk_owner_id=risk_owner_id,

        risk_co_owner_id=risk_owner_id,

        financial_year=DEFAULT_FIN_YEAR,

        risk_status=DEFAULT_RISK_STATUS,

        risk_progress="0",

        is_active=0

    )

    return risk

In [34]:

def create_risk_register(
    db,
    risk_data
):

    risk_id = generate_risk_id(
        db=db,
        dept_id=risk_data.dept_id
    )
    
    print("Generated Risk ID :", risk_id)

    db_risk = RiskRegister(

        **risk_data.model_dump(),

        risk_id=risk_id,

        created_by=1,

        created_on=datetime.utcnow(),

        is_deleted=0

    )

    db.add(db_risk)

    db.flush()

    db.refresh(db_risk)

    print(f"Created : {db_risk.risk_id}")

    return db_risk

In [35]:
def import_risk_registers(
    db,
    df,
    user_map
):

    risk_register_map = {}

    print("=" * 60)
    print("IMPORTING RISK REGISTER")
    print("=" * 60)

    for risk_no, risk_df in df.groupby("S.No"):

        print(f"\nRisk No : {risk_no}")

        row = risk_df.iloc[0]

        risk_schema = prepare_risk_register_data(
            db=db,
            row=row,
            user_map=user_map
        )

        risk = create_risk_register(
            db=db,
            risk_data=risk_schema
        )

        # risk_register_map[risk_no] = risk
        
        risk_register_map[risk_no] = {
            "risk_register_id": risk.risk_register_id,
            "risk_id": risk.risk_id
        }

    return risk_register_map

In [36]:
try:
    risk_register_map = import_risk_registers(
        db=db,
        df=df,
        user_map=user_map
    )

    db.commit()

    print("✅ Import completed successfully.")
    
except Exception as e:
    db.rollback()
    print(f"❌ Import failed: {e}")
    raise

IMPORTING RISK REGISTER

Risk No : 1.0
Generated Risk ID : HR-2392
Created : HR-2392

Risk No : 2.0
Generated Risk ID : HR-2393
Created : HR-2393

Risk No : 4.0
Generated Risk ID : HR-2394
Created : HR-2394

Risk No : 5.0
Generated Risk ID : HR-2395
Created : HR-2395
✅ Import completed successfully.


Risk Description

In [37]:
from datetime import datetime
from app.schemas.risk_description import RiskDescriptionCreate

def prepare_risk_description_data(
    row,
    risk_register_map
):

    # Parent Risk
    risk_register = risk_register_map[row["S.No"]]
    
    print(risk_register)


    # Inherent Risk
    inherent_likelihood, inherent_impact = parse_risk_code(
        row["Inherent Risk"]
    )

    # Current Risk
    current_likelihood, current_impact = parse_risk_code(
        row["Current Risk"]
    )

    return RiskDescriptionCreate(

        risk_register_id=risk_register["risk_register_id"],

        risk_id=risk_register["risk_id"],

        risk_description=clean_text(
            row["Risk Description"]
        ),

        inherent_risk_likelihood_id=inherent_likelihood,

        inherent_risk_impact_id=inherent_impact,

        mitigation=clean_text(
            row["Current Mitigation"]
        ),

        current_risk_likelihood_id=current_likelihood,

        current_risk_impact_id=current_impact

    )

In [38]:
# def create_risk_description(
#     db,
#     description_data
# ):

#     description = RiskDescription(

#         **description_data.model_dump(),

#         created_by=1,

#         created_on=datetime.utcnow(),

#         is_deleted=0

#     )

#     db.add(description)

#     db.flush()

#     db.refresh(description)

#     print(
#         f"Description ID : {description.risk_description_id}"
#     )

#     return description

In [39]:
def create_risk_description(
    db,
    description_data
):

    # Fetch parent Risk Register
    risk_register = db.query(RiskRegister).filter(
        RiskRegister.risk_register_id == description_data.risk_register_id,
        RiskRegister.is_deleted == 0
    ).first()

    if risk_register is None:
        raise Exception(
            f"Risk Register {description_data.risk_register_id} not found."
        )

    description = RiskDescription(

        risk_register_id=description_data.risk_register_id,

        # Fetch automatically from parent
        risk_id=risk_register.risk_id,

        risk_description=description_data.risk_description,

        inherent_risk_likelihood_id=description_data.inherent_risk_likelihood_id,

        inherent_risk_impact_id=description_data.inherent_risk_impact_id,

        mitigation=description_data.mitigation,

        current_risk_likelihood_id=description_data.current_risk_likelihood_id,

        current_risk_impact_id=description_data.current_risk_impact_id,

        created_by=1,

        created_on=datetime.utcnow(),

        is_deleted=0

    )

    db.add(description)

    db.flush()

    db.refresh(description)

    print(
        f"Created Description : {description.risk_description_id} ({description.risk_id})"
    )

    return description

In [40]:
def import_risk_descriptions(
    db,
    df,
    risk_register_map
):

    description_map = {}

    print("=" * 60)
    print("IMPORTING RISK DESCRIPTIONS")
    print("=" * 60)

    for index, row in df.iterrows():

        description_schema = prepare_risk_description_data(

            row=row,

            risk_register_map=risk_register_map

        )

        description = create_risk_description(

            db=db,

            description_data=description_schema

        )

        description_map[index] = {

            "risk_description_id": description.risk_description_id,

            "risk_register_id": description.risk_register_id,

            "risk_id": description.risk_id

        }

    return description_map

In [41]:
try:
    description_map = import_risk_descriptions(
        db=db,
        df=df,
        risk_register_map=risk_register_map)
    
    db.commit()
    print("✅ Import completed successfully.")
    
except Exception as e:
    db.rollback()
    print(f"❌ Import failed: {e}")
    raise

IMPORTING RISK DESCRIPTIONS
{'risk_register_id': 4558, 'risk_id': 'HR-2392'}
Created Description : 4181 (HR-2392)
{'risk_register_id': 4558, 'risk_id': 'HR-2392'}
Created Description : 4182 (HR-2392)
{'risk_register_id': 4558, 'risk_id': 'HR-2392'}
Created Description : 4183 (HR-2392)
{'risk_register_id': 4559, 'risk_id': 'HR-2393'}
Created Description : 4184 (HR-2393)
{'risk_register_id': 4560, 'risk_id': 'HR-2394'}
Created Description : 4185 (HR-2394)
{'risk_register_id': 4561, 'risk_id': 'HR-2395'}
Created Description : 4186 (HR-2395)
✅ Import completed successfully.


In [50]:
df

,S.No,Category,Risk Description,Inherent Risk,Current Mitigation,Current Risk,Risk Owner,Action Plan,Due Date,Action Owner,Q1,Q2,Q3,Q4,Backup
0,1.0,Recruitment & Hiring,Delay in acquisition of key Positions leading...,3D,1. Tie-ups with reputed recruitment agencies t...,2C,Shruti N.,,NaN,,NaN,NaN,NaN,NaN,NaN
1,1.0,Recruitment & Hiring,Inadequate background checks,3C,1. Continuous review of BGV's for each cases\n...,2C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
2,1.0,Recruitment & Hiring,Attrition risk : Inability to retain people at...,3D,"1. Retention planning for key talent, critical...",3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
3,2.0,Employee Exp.,Employee Disengagement and Inadequate talent D...,2C,1. SMART goals based Performance Management \...,2B,Abhijit S.,,NaN,,NaN,NaN,NaN,NaN,NaN
4,4.0,Complaiance Mgt.,Non-compliance to applicble labour laws leadin...,3D,1. All applicable labour compliances are mappe...,3C,Manali,,NaN,,NaN,NaN,NaN,NaN,NaN
5,5.0,Compensation & Benefits,Risk of overcompensation or undercompensation ...,3D,1. Periodic benchmark of compensation against ...,2C,Shirin V.,,NaN,,NaN,NaN,NaN,NaN,NaN


Risk Treatment

In [98]:
def has_treatment(row):

    action_plan = clean_text(row["Action Plan"])

    if action_plan is None:
        return False

    return True

In [99]:
def validate_treatment(
    row,
    user_map
):

    action_plan = clean_text(row["Action Plan"])

    if action_plan is None:
        return False, "Action Plan Missing"

    action_owner_name = clean_text(
        row["Action Owner"]
    )

    if action_owner_name is None:
        return False, "Action Owner Missing"

    action_owner = user_map.get(
        action_owner_name
    )

    if action_owner is None:
        return False, f"{action_owner_name} Not Found"

    return True, action_owner

In [100]:
import pandas as pd

def parse_due_date(value):

    if pd.isna(value):
        return None

    try:
        return pd.to_datetime(value).to_pydatetime()

    except:
        return None

In [93]:
DEFAULT_ACTION_STATUS = 1

In [101]:
from app.schemas.risk_treatment import RiskTreatmentCreate

def prepare_risk_treatment_data(

    row,

    description_map,

    action_owner

):

    description = description_map[row.name]

    return RiskTreatmentCreate(

        risk_description_id=description["risk_description_id"],

        risk_register_id=description["risk_register_id"],

        risk_id=description["risk_id"],

        action_plan=clean_text(
            row["Action Plan"]
        ),

        action_owner_id=action_owner["id"],

        target_date=parse_due_date(
            row["Due Date"]
        ),

        progress="0",

        action_status_id=DEFAULT_ACTION_STATUS,

        next_followup_date=None

    )

In [102]:
def find_risk_treatment(

    db,

    risk_description_id

):

    return db.query(RiskTreatment).filter(

        RiskTreatment.risk_description_id == risk_description_id,

        RiskTreatment.is_deleted == 0

    ).first()

In [104]:
from datetime import datetime

def create_risk_treatment(

    db,

    treatment_data

):

    treatment = RiskTreatment(

        **treatment_data.model_dump(),

        approval_status=0,

        created_by=1,

        created_on=datetime.utcnow(),

        is_deleted=0

    )

    db.add(treatment)

    db.flush()

    db.refresh(treatment)

    print(
        f"Created Treatment : {treatment.risk_treatment_id}"
    )

    return treatment

In [105]:
def import_risk_treatments(

    db,

    df,

    description_map,

    user_map

):

    treatment_map = {}

    print("=" * 60)
    print("IMPORTING RISK TREATMENTS")
    print("=" * 60)

    for index, row in df.iterrows():

        # --------------------------------------------------
        # Skip rows without Action Plan
        # --------------------------------------------------

        if not has_treatment(row):

            print(
                f"Skipping Row {index+1} (No Treatment)"
            )

            continue

        # --------------------------------------------------
        # Validate Treatment
        # --------------------------------------------------

        valid, result = validate_treatment(

            row,

            user_map

        )

        if not valid:

            print(
                f"Skipping Row {index+1} ({result})"
            )

            continue

        action_owner = result

        description = description_map[index]

        # --------------------------------------------------
        # Already Exists?
        # --------------------------------------------------

        treatment = find_risk_treatment(

            db=db,

            risk_description_id=description[
                "risk_description_id"
            ]

        )

        if treatment:

            print(
                f"Treatment Already Exists : {treatment.risk_treatment_id}"
            )

        else:

            treatment_schema = prepare_risk_treatment_data(

                row=row,

                description_map=description_map,

                action_owner=action_owner

            )

            treatment = create_risk_treatment(

                db=db,

                treatment_data=treatment_schema

            )

        treatment_map[index] = {

            "risk_treatment_id": treatment.risk_treatment_id

        }

    print("\n✅ Risk Treatment Import Completed.")

    return treatment_map

In [106]:
treatment_map = import_risk_treatments(

    db=db,

    df=df,

    description_map=description_map,

    user_map=user_map

)

IMPORTING RISK TREATMENTS
Skipping Row 1 ( Not Found)
Skipping Row 2 ( Not Found)
Skipping Row 3 ( Not Found)
Skipping Row 4 ( Not Found)
Skipping Row 5 ( Not Found)
Skipping Row 6 ( Not Found)

✅ Risk Treatment Import Completed.


In [9]:
from app.core.security import get_password_hash

print(get_password_hash("123456"))

ValidationError: 2 validation errors for Settings
FERNET_KEY
  Field required [type=missing, input_value={'DATABASE_URL': 'postgre...60', 'DB_SCHEMA': 'ers'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/missing
MAIN_URL
  Field required [type=missing, input_value={'DATABASE_URL': 'postgre...60', 'DB_SCHEMA': 'ers'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/missing

In [ ]:
from app.core.security import verify_password

hashed = "$2b$12$3mhfdz3aUpIPm5ZxD74PK.J.tJYV1pM3SDoHg47lkPcEKMKGqB43u"

print(verify_password("Mukesh123", hashed))

ValidationError: 2 validation errors for Settings
FERNET_KEY
  Field required [type=missing, input_value={'DATABASE_URL': 'postgre...60', 'DB_SCHEMA': 'ers'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/missing
MAIN_URL
  Field required [type=missing, input_value={'DATABASE_URL': 'postgre...60', 'DB_SCHEMA': 'ers'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.6/v/missing

: 

In [4]:
from cryptography.fernet import Fernet
print(Fernet.generate_key().decode())

wnbXcPc6SB_H2ZZd1VNdCt2ZZXYajkgjDNid6Wm5PCM=


In [2]:
from cryptography.fernet import Fernet

# 1. Generate a secure 32-byte key
key = Fernet.generate_key()
print(f"Secret Key: {key.decode()}")

# 2. Initialize the Fernet cipher suite
cipher_suite = Fernet(key)

# 3. Encrypt the data (data must be in bytes)
message = b"Top secret information"
token = cipher_suite.encrypt(message)
print(f"Encrypted Token: {token.decode()}")

# 4. Decrypt the token back to the original message
decrypted_message = cipher_suite.decrypt(token)
print(f"Decrypted Message: {decrypted_message.decode()}")


Secret Key: 6S-5O9Dv_O_7nN0yyhB28W_A1r17ZYXXq2AbVP4PFLA=
Encrypted Token: gAAAAABqVLtO9Dxqm8ycvInn1y7UzbNYo8CN8bOHfzSteid0ttnHhOeR6PzaNwcfAfHlBY068W-gpFOg78o_Hdwdl3SRm_O2HolgW-LAt46BikHLh89SCgA=
Decrypted Message: Top secret information
